# Retail Store Sales Insights and Demand Prediction System
**Academic internship project | Google Colab-ready**

This notebook analyzes the uploaded `Product-Sales-Region.xlsx` dataset. It inspects the actual schema before cleaning, documents cleaning decisions, performs exploratory and customer analysis, compares quantity-regression models, and evaluates a simple time-based demand forecast when the available chronology supports it.

**Important interpretation notes**
- `TotalPrice` is checked against `Quantity × UnitPrice × (1 − Discount)`; revenue calculations use the recorded `TotalPrice`.
- The dataset has `Product`, not a separate product-category field. Product-level analysis is therefore used.
- `CustomerName` is the available customer identifier. RFM results describe observed records only and should not be treated as a complete customer lifecycle history.
- Predictive models are evaluated on held-out data. Results are calculated by running this notebook; no results are pre-filled or fabricated.

## 1. Project Introduction

**Objective:** Understand sales performance, customer purchasing patterns, and observed demand, then test whether transaction quantity can be predicted from available transaction attributes. The analysis uses only the supplied workbook.

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 3. Load Dataset

In Google Colab, upload `Product-Sales-Region.xlsx` when prompted. If the file already exists at the expected path, it will be used directly. The notebook does not reload the exported cleaned CSV.

In [ ]:
from pathlib import Path

DATA_PATH = Path("/content/Product-Sales-Region.xlsx")
if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        excel_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
        if not excel_files:
            raise FileNotFoundError("Please upload the supplied Excel workbook (.xlsx or .xls).")
        DATA_PATH = Path(excel_files[0])
    except ImportError:
        # For local Jupyter, place the workbook in the current working directory.
        local_candidates = list(Path(".").glob("*.xlsx")) + list(Path(".").glob("*.xls"))
        if not local_candidates:
            raise FileNotFoundError("Place Product-Sales-Region.xlsx in the working directory.")
        DATA_PATH = local_candidates[0]

xls = pd.ExcelFile(DATA_PATH)
print("Workbook:", DATA_PATH.name)
print("Sheets:", xls.sheet_names)
raw_df = pd.read_excel(DATA_PATH, sheet_name=xls.sheet_names[0])
print("Loaded shape:", raw_df.shape)
display(raw_df.head())

## 4. Data Inspection

Review the actual columns, types, missing values, duplicate rows, numeric summaries, date ranges, and basic business-rule checks before deciding how to clean.

In [ ]:
print("Shape:", raw_df.shape)
print("\\nColumn names:")
print(raw_df.columns.tolist())
print("\\nData types:")
display(raw_df.dtypes.astype(str).rename("dtype").to_frame())
print("\\nMissing values:")
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Exact duplicate rows:", int(raw_df.duplicated().sum()))
display(raw_df.describe(include="all").T)

In [ ]:
# Inspect dates and likely business-rule issues only when the relevant columns exist.
for col in ["Date", "OrderDate", "DeliveryDate"]:
    if col in raw_df.columns:
        parsed = pd.to_datetime(raw_df[col], errors="coerce")
        print(f"{col}: parse failures={int(parsed.isna().sum())}, min={parsed.min()}, max={parsed.max()}")

if {"Quantity", "UnitPrice", "Discount", "TotalPrice"}.issubset(raw_df.columns):
    expected_total = raw_df["Quantity"] * raw_df["UnitPrice"] * (1 - raw_df["Discount"])
    print("Quantity <= 0:", int((raw_df["Quantity"] <= 0).sum()))
    print("UnitPrice < 0:", int((raw_df["UnitPrice"] < 0).sum()))
    print("Discount outside [0, 1]:", int((~raw_df["Discount"].between(0, 1)).sum()))
    print("TotalPrice < 0:", int((raw_df["TotalPrice"] < 0).sum()))
    print("TotalPrice formula mismatches (> 0.02 tolerance):",
          int(((raw_df["TotalPrice"] - expected_total).abs() > 0.02).sum()))

if {"OrderDate", "DeliveryDate"}.issubset(raw_df.columns):
    order_dates = pd.to_datetime(raw_df["OrderDate"], errors="coerce")
    delivery_dates = pd.to_datetime(raw_df["DeliveryDate"], errors="coerce")
    print("Delivery before order:", int((delivery_dates < order_dates).sum()))
if {"Date", "OrderDate"}.issubset(raw_df.columns):
    date_a = pd.to_datetime(raw_df["Date"], errors="coerce").dt.normalize()
    date_b = pd.to_datetime(raw_df["OrderDate"], errors="coerce").dt.normalize()
    print("Date/OrderDate disagreements:", int((date_a != date_b).sum()))

## 5. Data Cleaning

Cleaning is performed programmatically on a copy. Exact duplicate rows are removed. Date columns are parsed safely. Text values are trimmed. Missing promotions are labeled `No Promotion Recorded` so that absence is not confused with a known promotion. Rows with invalid essential identifiers/dates or impossible core numeric values are excluded from transaction analysis. Other missing values are retained for model-pipeline imputation or analysis-specific exclusion. Outliers are flagged with IQR rules for inspection, not automatically deleted.

In [ ]:
df = raw_df.copy()

# Normalize whitespace in text columns without altering meaningful labels.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

before_duplicates = len(df)
df = df.drop_duplicates().copy()
print("Exact duplicate rows removed:", before_duplicates - len(df))

# Parse all known date fields if present.
for col in ["Date", "OrderDate", "DeliveryDate"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# If Date is missing but OrderDate exists, use OrderDate; do not overwrite a valid Date.
if "Date" in df.columns and "OrderDate" in df.columns:
    df["Date"] = df["Date"].fillna(df["OrderDate"])

# Missing promotion is meaningful as unknown/no recorded promotion, not a fabricated promotion.
if "Promotion" in df.columns:
    df["Promotion"] = df["Promotion"].fillna("No Promotion Recorded")

# Exclude records with missing essential identifiers/date.
essential = [c for c in ["Date", "OrderID", "Product", "Quantity", "UnitPrice", "TotalPrice"] if c in df.columns]
before_essential = len(df)
df = df.dropna(subset=essential).copy()
print("Rows removed for missing essential fields:", before_essential - len(df))

# Enforce only clearly invalid core numeric values; no blanket outlier deletion.
for col in ["Quantity", "UnitPrice", "TotalPrice", "ShippingCost"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
invalid_mask = pd.Series(False, index=df.index)
if "Quantity" in df.columns:
    invalid_mask |= df["Quantity"].isna() | (df["Quantity"] <= 0)
if "UnitPrice" in df.columns:
    invalid_mask |= df["UnitPrice"].isna() | (df["UnitPrice"] < 0)
if "TotalPrice" in df.columns:
    invalid_mask |= df["TotalPrice"].isna() | (df["TotalPrice"] < 0)
if "Discount" in df.columns:
    df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")
    invalid_mask |= df["Discount"].notna() & ~df["Discount"].between(0, 1)
before_invalid = len(df)
df = df.loc[~invalid_mask].copy()
print("Rows removed for invalid core values:", before_invalid - len(df))

# Binary Returned validation, only where present.
if "Returned" in df.columns:
    df["Returned"] = pd.to_numeric(df["Returned"], errors="coerce")
    print("Returned values after parsing:", sorted(df["Returned"].dropna().unique().tolist()))

df = df.sort_values("Date").reset_index(drop=True)
print("Cleaned shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_after_cleaning"))
display(df.head())

In [ ]:
# IQR outlier audit for numeric columns; observations are flagged, not automatically removed.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_rows = []
for col in numeric_cols:
    series = df[col].dropna()
    if len(series) < 4:
        continue
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        lower, upper = q1, q3
    else:
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "column": col, "lower_IQR_fence": lower, "upper_IQR_fence": upper,
        "flagged_count": int(((series < lower) | (series > upper)).sum())
    })
outlier_audit = pd.DataFrame(outlier_rows)
display(outlier_audit)
print("Outliers are flagged for review, not dropped automatically, because high-value sales may be genuine.")

## 6. Exploratory Data Analysis (EDA)

Revenue is summarized from `TotalPrice`; transaction counts use unique `OrderID` where available. The workbook has no explicit product category or demographic columns, so product and available customer attributes are analyzed instead.

In [ ]:
# Dataset-dependent summary metrics
revenue = df["TotalPrice"].sum() if "TotalPrice" in df else np.nan
transactions = df["OrderID"].nunique() if "OrderID" in df else len(df)
quantity_sold = df["Quantity"].sum() if "Quantity" in df else np.nan
avg_transaction_value = revenue / transactions if transactions else np.nan
unique_customers = df["CustomerName"].nunique() if "CustomerName" in df else np.nan

summary = pd.DataFrame({
    "Metric": ["Total recorded revenue", "Transactions", "Quantity sold", "Average transaction value", "Unique customers"],
    "Value": [revenue, transactions, quantity_sold, avg_transaction_value, unique_customers]
})
display(summary)

In [ ]:
# Revenue and quantity by product
if "Product" in df.columns:
    product_perf = df.groupby("Product", dropna=False).agg(
        Revenue=("TotalPrice", "sum"),
        Quantity=("Quantity", "sum"),
        Transactions=("OrderID", "nunique") if "OrderID" in df.columns else ("Product", "size")
    ).sort_values("Revenue", ascending=False)
    display(product_perf)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    product_perf["Revenue"].sort_values().plot(kind="barh", ax=axes[0])
    axes[0].set_title("Revenue by Product")
    axes[0].set_xlabel("Recorded revenue")
    axes[0].set_ylabel("Product")
    product_perf["Quantity"].sort_values().plot(kind="barh", ax=axes[1])
    axes[1].set_title("Quantity Sold by Product")
    axes[1].set_xlabel("Quantity sold")
    axes[1].set_ylabel("Product")
    plt.tight_layout()
    plt.show()

In [ ]:
# Sales trends over time: daily and monthly aggregation
if "Date" in df.columns:
    daily = df.groupby(df["Date"].dt.floor("D")).agg(
        Revenue=("TotalPrice", "sum"), Quantity=("Quantity", "sum")
    ).sort_index()
    monthly = df.set_index("Date").resample("MS").agg(
        Revenue=("TotalPrice", "sum"), Quantity=("Quantity", "sum")
    )
    fig, axes = plt.subplots(2, 1, figsize=(14, 9))
    daily["Revenue"].plot(ax=axes[0], alpha=0.55, linewidth=1)
    axes[0].set_title("Daily Recorded Revenue")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Revenue")
    monthly["Revenue"].plot(ax=axes[1], marker="o")
    axes[1].set_title("Monthly Recorded Revenue")
    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Revenue")
    plt.tight_layout()
    plt.show()

    monthly["Quantity"].plot(figsize=(14, 4), marker="o")
    plt.title("Monthly Quantity Sold")
    plt.xlabel("Month")
    plt.ylabel("Quantity sold")
    plt.tight_layout()
    plt.show()

In [ ]:
# Regional and transaction attribute performance
for group_col in ["Region", "StoreLocation", "CustomerType", "PaymentMethod", "Promotion", "Salesperson"]:
    if group_col in df.columns:
        grouped = df.groupby(group_col, dropna=False).agg(
            Revenue=("TotalPrice", "sum"),
            Quantity=("Quantity", "sum"),
            Transactions=("OrderID", "nunique") if "OrderID" in df.columns else ("TotalPrice", "size")
        ).sort_values("Revenue", ascending=False)
        print(f"Performance by {group_col}")
        display(grouped)
        ax = grouped["Revenue"].sort_values().plot(kind="barh", figsize=(9, max(3, 0.35 * len(grouped))))
        ax.set_title(f"Recorded Revenue by {group_col}")
        ax.set_xlabel("Revenue")
        ax.set_ylabel(group_col)
        plt.tight_layout()
        plt.show()

In [ ]:
# Distribution and relationship checks
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["Quantity"], kde=True, ax=axes[0])
axes[0].set_title("Transaction Quantity Distribution")
axes[0].set_xlabel("Quantity")
sns.histplot(df["TotalPrice"], kde=True, ax=axes[1])
axes[1].set_title("Recorded Transaction Value Distribution")
axes[1].set_xlabel("TotalPrice")
sns.scatterplot(data=df, x="Quantity", y="TotalPrice", hue="Product" if "Product" in df.columns else None, ax=axes[2], legend=False)
axes[2].set_title("Quantity vs Recorded Transaction Value")
axes[2].set_xlabel("Quantity")
axes[2].set_ylabel("TotalPrice")
plt.tight_layout()
plt.show()

if "Returned" in df.columns:
    display(df.groupby("Returned").agg(
        Rows=("Returned", "size"),
        Revenue=("TotalPrice", "sum"),
        Quantity=("Quantity", "sum")
    ))

## 7. Customer Analysis

Customer-level metrics use the available `CustomerName` and `OrderID` fields. These are observed transactions, not a complete account-level history. Repeat versus one-time status is defined by the number of distinct order IDs in this dataset.

In [ ]:
if "CustomerName" in df.columns:
    customer_agg = {
        "Revenue": ("TotalPrice", "sum"),
        "Quantity": ("Quantity", "sum"),
        "Transactions": ("OrderID", "nunique") if "OrderID" in df.columns else ("TotalPrice", "size")
    }
    customer_summary = df.groupby("CustomerName", dropna=False).agg(**customer_agg)
    customer_summary["AverageOrderValue"] = customer_summary["Revenue"] / customer_summary["Transactions"].replace(0, np.nan)
    customer_summary["CustomerType"] = (
        df.groupby("CustomerName")["CustomerType"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
        if "CustomerType" in df.columns else pd.NA
    )
    customer_summary["PurchaseFrequencyGroup"] = np.where(customer_summary["Transactions"] > 1, "Repeat", "One-time")
    display(customer_summary.sort_values("Revenue", ascending=False).head(10).style.format({
        "Revenue": "{:,.2f}", "AverageOrderValue": "{:,.2f}"
    }))
    repeat_counts = customer_summary["PurchaseFrequencyGroup"].value_counts()
    display(repeat_counts.rename_axis("Customer group").to_frame("Customer count"))
    ax = repeat_counts.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Repeat vs One-Time Customers")
    ax.set_xlabel("Observed customer group")
    ax.set_ylabel("Number of customers")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    customer_summary["Revenue"].sort_values(ascending=False).head(10).sort_values().plot(kind="barh", figsize=(9, 5))
    plt.title("Top 10 Customers by Recorded Revenue")
    plt.xlabel("Revenue")
    plt.ylabel("Customer")
    plt.tight_layout()
    plt.show()
else:
    customer_summary = pd.DataFrame()
    print("CustomerName is not available; customer analysis is skipped.")

## 8. RFM Segmentation

RFM is calculated only if customer identity, date, order ID, and recorded value are available. Recency is measured against one day after the latest observed transaction date in this dataset. Frequency counts distinct orders; monetary value sums `TotalPrice`. Quantile scores are used only if enough distinct values exist; otherwise the notebook avoids forced segment labels.

In [ ]:
rfm = pd.DataFrame()
if {"CustomerName", "Date", "OrderID", "TotalPrice"}.issubset(df.columns) and not df.empty:
    reference_date = df["Date"].max().normalize() + pd.Timedelta(days=1)
    rfm = df.groupby("CustomerName").agg(
        LastPurchaseDate=("Date", "max"),
        Frequency=("OrderID", "nunique"),
        Monetary=("TotalPrice", "sum")
    )
    rfm["Recency"] = (reference_date - rfm["LastPurchaseDate"].dt.normalize()).dt.days
    rfm = rfm[["Recency", "Frequency", "Monetary", "LastPurchaseDate"]].sort_values("Monetary", ascending=False)
    display(rfm.describe(include="all"))

    # Rank-based scores tolerate tied/sparse values; labels are assigned only if data supports useful variation.
    if len(rfm) >= 10 and rfm["Recency"].nunique() >= 2 and rfm["Monetary"].nunique() >= 2:
        rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
        rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
        if rfm["Frequency"].nunique() >= 2:
            rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
        else:
            rfm["F_Score"] = 1

        def segment_customer(row):
            if row["R_Score"] >= 3 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
                return "High-value / loyal"
            if row["R_Score"] <= 2 and row["F_Score"] >= 2:
                return "At-risk repeat customer"
            if row["R_Score"] >= 3 and row["F_Score"] <= 2:
                return "Recent / developing"
            return "Other observed customers"

        rfm["Segment"] = rfm.apply(segment_customer, axis=1)
        display(rfm.groupby("Segment").agg(
            Customers=("Monetary", "size"),
            MedianRecencyDays=("Recency", "median"),
            MeanFrequency=("Frequency", "mean"),
            TotalMonetary=("Monetary", "sum")
        ).sort_values("TotalMonetary", ascending=False))
        ax = rfm["Segment"].value_counts().plot(kind="bar", figsize=(9, 4))
        ax.set_title("Observed Customer RFM Segments")
        ax.set_xlabel("Segment")
        ax.set_ylabel("Customers")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("RFM metrics are available, but the customer sample/variation is insufficient for stable quantile segments.")
else:
    print("RFM skipped: required customer/date/order/value columns are not all available.")

## 9. Feature Engineering

Transaction-level features are derived only from columns available in this workbook. Customer aggregates are kept for descriptive analysis and are **not** joined into the transaction prediction features, avoiding leakage from full-period customer behavior. Date-derived features are based on each transaction date.

In [ ]:
model_df = df.copy()
if "Date" in model_df.columns:
    model_df["Year"] = model_df["Date"].dt.year
    model_df["Month"] = model_df["Date"].dt.month
    model_df["DayOfWeek"] = model_df["Date"].dt.dayofweek
    model_df["DayOfYear"] = model_df["Date"].dt.dayofyear
    model_df["Quarter"] = model_df["Date"].dt.quarter
    model_df["IsWeekend"] = (model_df["Date"].dt.dayofweek >= 5).astype(int)

# Useful transaction-derived fields for descriptive analysis (not used as predictors of Quantity).
if {"Quantity", "UnitPrice"}.issubset(model_df.columns):
    model_df["GrossLineValue"] = model_df["Quantity"] * model_df["UnitPrice"]
if {"TotalPrice", "Quantity"}.issubset(model_df.columns):
    model_df["RecordedValuePerUnit"] = model_df["TotalPrice"] / model_df["Quantity"].replace(0, np.nan)

if not customer_summary.empty:
    customer_features = customer_summary[["Revenue", "Transactions", "Quantity", "AverageOrderValue"]].rename(columns={
        "Revenue": "CustomerTotalRevenue",
        "Transactions": "CustomerFrequency",
        "Quantity": "CustomerTotalQuantity",
        "AverageOrderValue": "CustomerAverageOrderValue"
    })
    print("Customer-level features prepared for descriptive use only; not joined into quantity model to avoid full-period leakage.")
display(model_df.head())

## 10. Predictive Modeling

**Target:** transaction `Quantity`. This is a direct demand-related target available in the data. Predictors exclude `TotalPrice`, `GrossLineValue`, and other fields directly derived from the target to avoid target leakage. The split is chronological where usable dates exist; otherwise a reproducible random split is used and reported. Missing predictors are imputed inside sklearn pipelines.

In [ ]:
target = "Quantity"
if target not in df.columns:
    raise ValueError("Quantity target is not available; regression modeling cannot proceed.")

# Use features known from the transaction record; exclude direct target-derived fields and identifiers.
candidate_features = [
    "Product", "Region", "StoreLocation", "CustomerType", "UnitPrice", "Discount",
    "Salesperson", "PaymentMethod", "Promotion", "Returned", "ShippingCost",
    "Year", "Month", "DayOfWeek", "DayOfYear", "Quarter", "IsWeekend"
]
feature_cols = [c for c in candidate_features if c in model_df.columns]
# Date itself and ID/name columns are excluded. TotalPrice and any engineered target-derived values are excluded.
feature_cols = [c for c in feature_cols if c not in ["TotalPrice", "GrossLineValue", "RecordedValuePerUnit", "OrderID", "CustomerName", "Date", "OrderDate", "DeliveryDate"]]
if not feature_cols:
    raise ValueError("No usable predictors are available in the actual dataset.")

X = model_df[feature_cols].copy()
y = pd.to_numeric(model_df[target], errors="coerce")
valid_rows = y.notna()
X, y = X.loc[valid_rows], y.loc[valid_rows]
date_for_split = model_df.loc[valid_rows, "Date"] if "Date" in model_df.columns else None

# Chronological split is preferred to better reflect prediction on later transactions.
if date_for_split is not None and date_for_split.notna().sum() == len(y) and date_for_split.nunique() >= 2:
    order = np.argsort(date_for_split.to_numpy())
    split_at = int(len(order) * 0.8)
    split_at = min(max(split_at, 1), len(order) - 1)
    train_idx, test_idx = order[:split_at], order[split_at:]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    split_method = "Chronological 80/20 split by transaction date"
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    split_method = "Random 80/20 split (date unavailable or unusable)"

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2, n_jobs=-1
    )
}
fitted_models = {}
predictions = {}
evaluation_rows = []

print("Target:", target)
print("Features:", feature_cols)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Split method:", split_method)

for name, estimator in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name] = pipe
    predictions[name] = pred
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    evaluation_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": rmse,
        "R2": r2_score(y_test, pred)
    })

model_comparison = pd.DataFrame(evaluation_rows).sort_values("RMSE").reset_index(drop=True)
best_model_name = model_comparison.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_predictions = predictions[best_model_name]
print("Best model selected by lowest holdout RMSE:", best_model_name)
display(model_comparison)

## 11. Model Evaluation

The model comparison reports MAE, RMSE, and R² on the same holdout. The model with the lowest holdout RMSE is selected for subsequent interpretation. Performance is descriptive of this dataset and split, not a guarantee of future accuracy.

In [ ]:
# Plot actual versus predicted values for both regressors.
fig, axes = plt.subplots(1, len(predictions), figsize=(7 * len(predictions), 5))
if len(predictions) == 1:
    axes = [axes]
for ax, (name, pred) in zip(axes, predictions.items()):
    sns.scatterplot(x=y_test, y=pred, alpha=0.65, ax=ax)
    low = min(float(y_test.min()), float(np.min(pred)))
    high = max(float(y_test.max()), float(np.max(pred)))
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_title(f"Actual vs Predicted Quantity: {name}")
    ax.set_xlabel("Actual quantity")
    ax.set_ylabel("Predicted quantity")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "ActualQuantity": y_test.to_numpy(),
    "BestModelPredictedQuantity": best_predictions
}).head(15))

## 12. Demand Analysis / Prediction

Demand is aggregated by date and by product. For a simple forecast, daily total quantity is modeled with lagged demand and calendar features and evaluated on a chronological holdout. This is a baseline forecasting exercise, not a claim of sophisticated seasonality modeling. Days without recorded transactions are filled with zero only inside the date-indexed demand series to create a continuous calendar series; this assumption is stated explicitly.

In [ ]:
# Demand patterns by product over time
if {"Date", "Product", "Quantity"}.issubset(df.columns):
    product_daily = df.groupby([pd.Grouper(key="Date", freq="MS"), "Product"])["Quantity"].sum().reset_index()
    pivot_product = product_daily.pivot(index="Date", columns="Product", values="Quantity").fillna(0)
    ax = pivot_product.plot(figsize=(14, 6), marker="o")
    ax.set_title("Monthly Quantity Sold by Product")
    ax.set_xlabel("Month")
    ax.set_ylabel("Quantity sold")
    plt.tight_layout()
    plt.show()

if {"Date", "Quantity"}.issubset(df.columns):
    daily_demand = df.groupby("Date")["Quantity"].sum().sort_index()
    if len(daily_demand.index) >= 60 and daily_demand.index.notna().all():
        # Reindex to a daily calendar. Missing dates mean no transaction was recorded that day.
        daily_demand = daily_demand.asfreq("D", fill_value=0)
        demand_frame = daily_demand.rename("Demand").to_frame()
        for lag in [1, 7, 14, 28]:
            demand_frame[f"lag_{lag}"] = demand_frame["Demand"].shift(lag)
        demand_frame["rolling_mean_7"] = demand_frame["Demand"].shift(1).rolling(7).mean()
        demand_frame["rolling_mean_28"] = demand_frame["Demand"].shift(1).rolling(28).mean()
        demand_frame["day_of_week"] = demand_frame.index.dayofweek
        demand_frame["month"] = demand_frame.index.month
        demand_frame["day_of_year"] = demand_frame.index.dayofyear
        demand_frame = demand_frame.dropna()

        if len(demand_frame) >= 40:
            split_point = int(len(demand_frame) * 0.8)
            train_demand = demand_frame.iloc[:split_point]
            test_demand = demand_frame.iloc[split_point:]
            demand_features = [c for c in demand_frame.columns if c != "Demand"]

            demand_model = RandomForestRegressor(
                n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=3, n_jobs=-1
            )
            demand_model.fit(train_demand[demand_features], train_demand["Demand"])
            demand_pred = demand_model.predict(test_demand[demand_features])

            demand_metrics = pd.DataFrame([{
                "Model": "Random Forest lag-feature daily demand baseline",
                "MAE": mean_absolute_error(test_demand["Demand"], demand_pred),
                "RMSE": np.sqrt(mean_squared_error(test_demand["Demand"], demand_pred)),
                "R2": r2_score(test_demand["Demand"], demand_pred),
                "Train days": len(train_demand),
                "Test days": len(test_demand)
            }])
            display(demand_metrics)

            plt.figure(figsize=(14, 5))
            plt.plot(test_demand.index, test_demand["Demand"], label="Actual daily quantity")
            plt.plot(test_demand.index, demand_pred, label="Predicted daily quantity")
            plt.title("Daily Demand: Chronological Holdout")
            plt.xlabel("Date")
            plt.ylabel("Quantity sold")
            plt.legend()
            plt.tight_layout()
            plt.show()

            demand_forecast_results = pd.DataFrame({
                "Date": test_demand.index,
                "ActualDemand": test_demand["Demand"].to_numpy(),
                "PredictedDemand": demand_pred
            })
            display(demand_forecast_results.head(15))
        else:
            print("Insufficient rows after lag creation for a reliable demand holdout; forecast skipped.")
            demand_metrics = pd.DataFrame()
            demand_model = None
            demand_forecast_results = pd.DataFrame()
    else:
        print("Insufficient chronological observations for demand forecasting; forecast skipped.")
        demand_metrics = pd.DataFrame()
        demand_model = None
        demand_forecast_results = pd.DataFrame()
else:
    print("Date or Quantity is unavailable; demand forecast skipped.")
    demand_metrics = pd.DataFrame()
    demand_model = None
    demand_forecast_results = pd.DataFrame()

## 13. Feature Importance

For the best tree-based transaction model, display impurity-based feature importance after preprocessing. The values indicate predictive association in this fitted model, not causal effects. If Linear Regression wins the comparison, the Random Forest is still shown as a tree-based reference model, with its holdout performance stated alongside the feature importance.

In [ ]:
tree_pipe = fitted_models.get("Random Forest Regressor")
if tree_pipe is not None:
    fitted_preprocessor = tree_pipe.named_steps["preprocessor"]
    rf_estimator = tree_pipe.named_steps["model"]
    try:
        transformed_names = fitted_preprocessor.get_feature_names_out()
        importances = pd.Series(rf_estimator.feature_importances_, index=transformed_names).sort_values(ascending=False)
        display(importances.head(20).rename("Importance").to_frame())
        importances.head(20).sort_values().plot(kind="barh", figsize=(10, 7))
        plt.title("Random Forest Feature Importance for Transaction Quantity")
        plt.xlabel("Importance")
        plt.ylabel("Encoded feature")
        plt.tight_layout()
        plt.show()
        print("Interpretation: importance describes features associated with predictions in this model; it does not establish causation.")
    except Exception as exc:
        print("Feature importance could not be mapped to encoded feature names:", exc)
else:
    print("Tree-based model was not fitted; feature importance unavailable.")

## 14. Business Insights

The following cells generate findings from the cleaned data and model outputs at runtime. They avoid hard-coded conclusions and recommendations.

In [ ]:
insights = []
if "TotalPrice" in df.columns and "Product" in df.columns:
    product_revenue = df.groupby("Product")["TotalPrice"].sum().sort_values(ascending=False)
    if len(product_revenue):
        insights.append(f"Highest recorded-revenue product: {product_revenue.index[0]} ({product_revenue.iloc[0]:,.2f} revenue units).")
if "Quantity" in df.columns and "Product" in df.columns:
    product_quantity = df.groupby("Product")["Quantity"].sum().sort_values(ascending=False)
    if len(product_quantity):
        insights.append(f"Highest quantity product: {product_quantity.index[0]} ({product_quantity.iloc[0]:,.0f} units).")
if "Region" in df.columns:
    region_revenue = df.groupby("Region")["TotalPrice"].sum().sort_values(ascending=False)
    if len(region_revenue):
        insights.append(f"Highest recorded-revenue region: {region_revenue.index[0]} ({region_revenue.iloc[0]:,.2f}).")
if "CustomerName" in df.columns and "OrderID" in df.columns:
    freq = df.groupby("CustomerName")["OrderID"].nunique()
    repeat_pct = (freq.gt(1).mean() * 100) if len(freq) else np.nan
    insights.append(f"Customers with multiple observed orders: {int(freq.gt(1).sum())} of {len(freq)} ({repeat_pct:.1f}%).")
if not model_comparison.empty:
    best_row = model_comparison.iloc[0]
    insights.append(f"Lowest holdout RMSE transaction-quantity model: {best_row['Model']} (MAE={best_row['MAE']:.3f}, RMSE={best_row['RMSE']:.3f}, R²={best_row['R2']:.3f}).")
if isinstance(demand_metrics, pd.DataFrame) and not demand_metrics.empty:
    dm = demand_metrics.iloc[0]
    insights.append(f"Daily demand baseline holdout: MAE={dm['MAE']:.3f}, RMSE={dm['RMSE']:.3f}, R²={dm['R2']:.3f}.")
if "Promotion" in df.columns and "TotalPrice" in df.columns:
    promo = df.groupby("Promotion")["TotalPrice"].agg(["sum", "count"]).sort_values("sum", ascending=False)
    if len(promo):
        insights.append(f"Promotion revenue totals are descriptive and reflect differences in transaction mix; they do not prove promotional impact.")

print("DATA-DRIVEN FINDINGS")
for i, item in enumerate(insights, 1):
    print(f"{i}. {item}")

print("\\nBUSINESS RECOMMENDATIONS")
recommendations = [
    "Use product-level quantity and revenue trends to inform replenishment discussions, while validating inventory and stockout data before setting reorder quantities.",
    "Review repeat-customer behavior as a small observed segment; avoid treating one-time status as churn without a defined customer lifecycle window.",
    "Use the holdout model as a baseline only. Compare against simple baselines and monitor performance on future data before operational use.",
    "Investigate high-value products, regions, and transaction attributes with operational context; descriptive associations do not establish causation.",
    "Collect stock-on-hand, stockout, product-category, and richer customer-history data if future work requires inventory optimization, category analysis, or churn prediction."
]
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

## 15. Export Outputs

Exports the cleaned DataFrame and selected analysis tables for submission. The notebook continues to use `df` in memory and does not reload the exported CSV.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path(".")
cleaned_path = OUTPUT_DIR / "cleaned_retail_sales.csv"
df.to_csv(cleaned_path, index=False)
print("Cleaned dataset exported to:", cleaned_path.resolve())

# Optional submission artifacts
if not model_comparison.empty:
    model_comparison.to_csv(OUTPUT_DIR / "quantity_model_comparison.csv", index=False)
    print("Model comparison exported to:", (OUTPUT_DIR / "quantity_model_comparison.csv").resolve())
if isinstance(demand_metrics, pd.DataFrame) and not demand_metrics.empty:
    demand_metrics.to_csv(OUTPUT_DIR / "demand_forecast_metrics.csv", index=False)
    demand_forecast_results.to_csv(OUTPUT_DIR / "demand_forecast_holdout.csv", index=False)
    print("Demand forecast outputs exported to:", OUTPUT_DIR.resolve())

print("Final cleaned DataFrame shape:", df.shape)
print("Notebook completed. Review printed findings and holdout metrics before including them in the project report.")